In [ ]:
import geopandas as gpd

# Load the cross-validation file
file_path = 'predicted_shorelines_cut_crossval.geojson'
gdf = gpd.read_file(file_path)

# Print the first 5 rows and the column names to identify the 'fold' identifier
print("--- Columns available ---")
print(gdf.columns)

print("\n--- First 5 rows to check values ---")
print(gdf.head())

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

# ==========================================
# 1. HELPER FUNCTIONS (Reusable logic)
# ==========================================

def load_file(file_path):
    """
    Safe file loading wrapper.
    """
    print(f"Please wait, loading: {file_path} ...")
    try:
        return gpd.read_file(file_path)
    except Exception as e:
        print(f"!! Error loading file: {e}")
        return None

def count_points(geom):
    """
    Counts the number of points/vertices in a geometry.
    Supports MultiPoint (shorelines) and LineString.
    """
    if geom is None:
        return 0
    if geom.geom_type == 'MultiPoint':
        return len(geom.geoms)
    elif geom.geom_type == 'LineString':
        return len(geom.coords)
    else:
        return 0

def display_metrics(df, label, indent=""):
    """
    Internal helper to print the actual numbers.
    Used for both Global stats and Per-Site stats.
    """
    num_lines = len(df)
    mean_pts = df['point_count'].mean()
    max_pts = df['point_count'].max()
    min_pts = df['point_count'].min()
    std_pts = df['point_count'].std()

    print(f"{indent}--- {label} ---")
    print(f"{indent}Number of Shorelines (N): {num_lines}")
    print(f"{indent}Average Points per Line : {mean_pts:.2f}")
    print(f"{indent}Max Points in a Line    : {max_pts}")
    print(f"{indent}Min Points in a Line    : {min_pts}")
    print(f"{indent}Standard Deviation (SD) : {std_pts:.2f}")
    print(f"{indent}" + "-" * 30)

def print_dataset_stats(gdf, main_title):
    """
    Calculates and prints stats:
    1. Globally (Total for this selection)
    2. Broken down by 'site' (if available)
    """
    print(f"\n>>> ANALYZING: {main_title}")
    print("=" * 60)
    
    if gdf.empty:
        print(">> Warning: No data found for this selection.\n")
        return

    # 1. Pre-calculate points for the whole set (Efficiency)
    # We use copy() to avoid SettingWithCopyWarning
    df_calc = gdf.copy()
    df_calc['point_count'] = df_calc.geometry.apply(count_points)

    # 2. Display GLOBAL stats
    display_metrics(df_calc, "GLOBAL STATS (All sites combined)")

    # 3. Display PER-SITE stats
    if 'site' in df_calc.columns:
        # Get unique sites sorted alphabetically
        unique_sites = sorted(df_calc['site'].unique())
        
        if len(unique_sites) > 0:
            print(f"\n   [ BREAKDOWN BY SITE ({len(unique_sites)} found) ]")
            
            for site in unique_sites:
                # Filter data for this specific site
                site_subset = df_calc[df_calc['site'] == site]
                
                # Display stats with indentation for readability
                display_metrics(site_subset, f"SITE: {site.upper()}", indent="   ")
    else:
        print("   (No 'site' column found, skipping site breakdown)")
    
    print("=" * 60 + "\n")

def load_file(file_path):
    """
    Safe file loading wrapper.
    """
    print(f"Please wait, loading: {file_path} ...")
    try:
        return gpd.read_file(file_path)
    except Exception as e:
        print(f"!! Error loading file: {e}")
        return None

# ==========================================
# 2. ANALYSIS WORKFLOWS
# ==========================================

def analyze_baseline(file_path):
    """
    Workflow for the Baseline file (analyzes the whole file).
    """
    print(f"\n******************************************")
    print(f" PROCESSING FILE: BASELINE")
    print(f"******************************************")
    
    gdf = load_file(file_path)
    if gdf is not None:
        print_dataset_stats(gdf, "BASELINE FULL DATASET")

def analyze_experiment(gdf, experiment_name):
    """
    Workflow for Crossval file:
    1. Filters by specific Fold (experimentname).
    2. Splits by Image Type (Oblique vs Rectified).
    """
    print(f"\n******************************************")
    print(f" EXPERIMENT NAME: {experiment_name}")
    print(f"******************************************")
    
    # gdf = load_file(file_path)
    if gdf is None:
        return None

    # 1. Filter by Fold
    if 'experimentname' not in gdf.columns:
        print("Error: 'experimentname' column missing.")
        return

    experiment_subset = gdf[gdf['experimentname'] == experiment_name].copy()
    
    if experiment_subset.empty:
        print(f"Error: No data found for experiment '{experiment_name}'. Check spelling.")
        return

    # 2. Split by Image Type
    if 'image_type' in experiment_subset.columns:
        # A) OBLIQUE
        oblique = experiment_subset[experiment_subset['image_type'] == 'oblique']
        print_dataset_stats(oblique, f"EXPERIMENT: {experiment_name} | TYPE: OBLIQUE")
        
        # B) RECTIFIED
        rectified = experiment_subset[experiment_subset['image_type'] == 'rectified']
        print_dataset_stats(rectified, f"EXPERIMENT: {experiment_name} | TYPE: RECTIFIED")
    else:
        print("Warning: 'image_type' column missing. Analyzing total fold instead.")
        print_dataset_stats(experiment_subset, f"EXPERIMENT: {experiment_name} (Total)")

    return experiment_subset

def find_exclusive_dates(baseline_gdf, fold_gdf, fold_name):
    """
    Identifies specific IMAGES (Date + Time) that exist 
    in the DeepLab Fold but are MISSING in the Baseline.
    """
    print(f"\n******************************************")
    print(f" COMPARING: DeepLab ({fold_name}) vs BASELINE")
    print(f" Objective: Find specific timestamps in DeepLab NOT in Baseline")
    print(f"******************************************")

    if baseline_gdf is None or fold_gdf is None:
        print("Error: One of the datasets is missing. Cannot compare.")
        return

    # 1. Keep full datetime (Do NOT strip time) to distinguish multiple images per day
    # We convert to string to ensure easy set comparison
    baseline_gdf['timestamp'] = pd.to_datetime(baseline_gdf['date'], errors='coerce')
    fold_gdf['timestamp'] = pd.to_datetime(fold_gdf['date'], errors='coerce')

    # 2. Create Unique IDs: Tuple (Site, Timestamp)
    base_pairs = set(zip(baseline_gdf['site'], baseline_gdf['timestamp']))
    fold_pairs = set(zip(fold_gdf['site'], fold_gdf['timestamp']))

    # 3. Find Difference: Items in Fold NOT in Base
    exclusive_to_deeplab = sorted(list(fold_pairs - base_pairs))

    # 4. Print Results
    count_diff = len(exclusive_to_deeplab)
    
    if count_diff == 0:
        print(">> RESULT: No extra images found. All DeepLab images exist in Baseline.")
    else:
        print(f">> RESULT: Found {count_diff} shorelines exclusive to DeepLab:\n")
        
        # Convert to DataFrame for grouping
        diff_df = pd.DataFrame(exclusive_to_deeplab, columns=['site', 'ts'])
        
        for site in sorted(diff_df['site'].unique()):
            print(f"   [ SITE: {site.upper()} ]")
            timestamps = diff_df[diff_df['site'] == site]['ts']
            for ts in timestamps:
                # Print full timestamp
                print(f"     - {ts}")
            print("")

    

In [ ]:
# Baseline
FILE_BASELINE = 'predicted_shorelines_cut_baseline.geojson'

# 1. Run Baseline Analysis
analyze_baseline(FILE_BASELINE)

In [ ]:

# --- CONFIGURATION ---
FILE_EXPERIMENT1_3 = 'predicted_shorelines_cut_expermient1-3.geojson'
# FILE_CROSSVAL = 'predicted_shorelines_cut_crossval.geojson'

# --- RUN ---
TARGETS_EXPERIMENT_NAME = [
    "exp1_bilstm",
    "exp2_unet",
    "exp2_attunet",
    "exp2_deeplabv3",
    "exp2_ducknet"
]

gdf_experiment1_3 = load_file(FILE_EXPERIMENT1_3)

for target_experiment in TARGETS_EXPERIMENT_NAME:
    analyze_experiment(gdf_experiment1_3, target_experiment)

In [ ]:
# --- CONFIGURATION ---
FILE_EXPERIMENT_CROSSVAL = 'predicted_shorelines_cut_crossval.geojson'
# FILE_CROSSVAL = 'predicted_shorelines_cut_crossval.geojson'

# --- RUN ---
TARGETS_EXPERIMENT_NAME = [
    "fold0_DeepLabV3512x512",
    "fold1_DeepLabV3512x512",
    "fold2_DeepLabV3512x512",
    "fold3_DeepLabV3512x512",
    "fold4_DeepLabV3512x512"
]

gdf_experiment_crossval = load_file(FILE_EXPERIMENT_CROSSVAL)

for target_experiment in TARGETS_EXPERIMENT_NAME:
    analyze_experiment(gdf_experiment_crossval, target_experiment)

In [ ]:
# --- CONFIGURATION ---
FILE_BASELINE = 'predicted_shorelines_cut_baseline.geojson'
FILE_CROSSVAL = 'predicted_shorelines_cut_crossval.geojson'

TARGET_FOLD_NAME = "fold1_DeepLabV3512x512" 

# --- RUN ---

# Load files ONCE to be efficient
gdf_baseline = load_file(FILE_BASELINE)
gdf_crossval = load_file(FILE_CROSSVAL)

# 1. Run Baseline Analysis
analyze_baseline(FILE_BASELINE)

# 2. Run Crossval Analysis and get the subset
gdf_fold_subset = analyze_experiment(gdf_crossval, TARGET_FOLD_NAME)

analyze_experiment(gdf_experiment1_3, target_experiment)

# 3. Run Comparison (New Feature)
if gdf_baseline is not None and gdf_fold_subset is not None:
    find_exclusive_dates(gdf_baseline, gdf_fold_subset, TARGET_FOLD_NAME)


In [ ]:
def find_missing_in_oblique(gdf, experiment_name):
    """
    Checks consistency within an experiment.
    Finds which image exists in RECTIFIED but is missing in OBLIQUE.
    """
    print(f"\n******************************************")
    print(f" CONSISTENCY CHECK: {experiment_name}")
    print(f" Objective: Find missing Oblique shoreline")
    print(f"******************************************")

    if gdf is None:
        print("Error: Dataset is None.")
        return

    # 1. Filter by Experiment
    exp_subset = gdf[gdf['experimentname'] == experiment_name].copy()
    
    if exp_subset.empty:
        print(f"Error: No data found for {experiment_name}")
        return

    # 2. Split into the two types
    oblique = exp_subset[exp_subset['image_type'] == 'oblique']
    rectified = exp_subset[exp_subset['image_type'] == 'rectified']

    print(f"   > Rectified count: {len(rectified)}")
    print(f"   > Oblique count  : {len(oblique)}")

    if len(oblique) >= len(rectified):
        print("   > No missing oblique images detected (Count is equal or higher).")
        return

    # 3. Create Unique IDs (Site + Timestamp)
    # We use exact timestamp string matching or datetime objects
    rectified['ts'] = pd.to_datetime(rectified['date'], errors='coerce')
    oblique['ts'] = pd.to_datetime(oblique['date'], errors='coerce')

    rect_ids = set(zip(rectified['site'], rectified['ts']))
    obl_ids = set(zip(oblique['site'], oblique['ts']))

    # 4. Find what is in Rectified but NOT in Oblique
    missing_in_obl = sorted(list(rect_ids - obl_ids))

    # 5. Output
    print(f"\n>> RESULT: Found {len(missing_in_obl)} image(s) missing in Oblique view:\n")
    
    # Convert to DataFrame for display
    if missing_in_obl:
        diff_df = pd.DataFrame(missing_in_obl, columns=['site', 'ts'])
        
        for site in sorted(diff_df['site'].unique()):
            print(f"   [ SITE: {site.upper()} ]")
            timestamps = diff_df[diff_df['site'] == site]['ts']
            for ts in timestamps:
                print(f"     - {ts}")
            print("")

# --- CONFIGURATION ---
FILE_EXPERIMENT1_3 = 'predicted_shorelines_cut_expermient1-3.geojson'
FILE_CROSSVAL = 'predicted_shorelines_cut_crossval.geojson'

TARGET_FOLD_NAME = "fold1_DeepLabV3512x512" 

# --- RUN ---

# Load files ONCE to be efficient
gdf_experiment1_3 = load_file(FILE_EXPERIMENT1_3)
gdf_crossval = load_file(FILE_CROSSVAL)

# 1. Run Baseline Analysis
find_missing_in_oblique(gdf_experiment1_3, "exp1_bilstm")

Please wait, loading: predicted_shorelines_cut_expermient1-3.geojson ...
Please wait, loading: predicted_shorelines_cut_crossval.geojson ...

******************************************
 CONSISTENCY CHECK: exp1_bilstm
 Objective: Find missing Oblique shoreline
******************************************
   > Rectified count: 174
   > Oblique count  : 173

>> RESULT: Found 1 image(s) missing in Oblique view:

   [ SITE: CIES ]
     - 2019-06-24 12:54:34+00:00



/home/josep/miniconda/envs/shoreline-extraction/lib/python3.13/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/home/josep/miniconda/envs/shoreline-extraction/lib/python3.13/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
